In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "HIDDEN"

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 371.3 kB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [ ]:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """This function fetches the currency conversion factor between given base currency & target currency"""
  url = f'https://v6.exchangerate-api.com/v6/KEY/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """Given a currency conversion rate and base currency value, this function calculates the target currency value"""
  return base_currency_value*conversion_rate


In [ ]:
get_conversion_factor.invoke({'base_currency': 'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1752537601,
 'time_last_update_utc': 'Tue, 15 Jul 2025 00:00:01 +0000',
 'time_next_update_unix': 1752624001,
 'time_next_update_utc': 'Wed, 16 Jul 2025 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 86.0194}

In [ ]:
convert.invoke({'base_currency_value': 50, 'conversion_rate': 86.01})

4300.5

In [ ]:
llm = ChatOpenAI()

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion rate of USD to INR, and can you convert 50 USD to INR')]

In [ ]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_vZVPZCYxraXztOPkle1pFs16', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_JnjXXAk1NsNbELELsfCUXWBZ', 'function': {'arguments': '{"base_currency_value": 50}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 116, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BtdE4jXDOu1CPchf0uMWzGxCWv0VU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--ff2d0a50-e1b8-4e4f-a6d6-9e89f25a3ab4-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_cur

In [ ]:
#llm_with_tools.invoke(messages).tool_calls

In [ ]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_Kk87laQ9PSfKry8BqB5DoEjj',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 50},
  'id': 'call_IAd5w9ENsFK9VS25qjb6MPUB',
  'type': 'tool_call'}]

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  #execute the first tool and get the conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 =get_conversion_factor.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)

In [ ]:
messages

[HumanMessage(content='What is the conversion rate of USD to INR, and can you convert 50 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_l5wfaC1a6kKUtUdIz7ct0VKs', 'function': {'arguments': '{"base_currency": "USD", "target_currency": "INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_7XNYHgakJm1tGC2siU5g2C0D', 'function': {'arguments': '{"base_currency_value": 50}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 116, 'total_tokens': 168, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BtdE8IYAQzyztYMhW2cjRtpzn01EW', 'service_tier': 'default', 'finish_reason':

In [ ]:
llm_with_tools.invoke(messages)

AIMessage(content='The conversion rate of USD to INR is 86.0194. \n\nConverting 50 USD to INR gives 4300.97 INR.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 328, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BtdEwrARzL6wmaZjJra8d7PTbpy7l', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--e79d2060-54d0-4859-8243-eb022e1bd4ea-0', usage_metadata={'input_tokens': 328, 'output_tokens': 34, 'total_tokens': 362, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
#Question: What is the conversion rate of USD to INR, and can you convert 50 USD to INR
#Answer: The conversion rate of USD to INR is 86.0194. \n\nConverting 50 USD to INR gives 4300.97 INR.